In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import pickle
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed_dir = project_root / 'data' / 'processed'

candidates = pd.read_csv(processed_dir / 'candidates_clean.csv')
jobs = pd.read_csv(processed_dir / 'jobs_clean.csv')

# ══════════════════════════════════════════════════════════════════
# TF-IDF ON SKILLS
# ══════════════════════════════════════════════════════════════════
# We combine candidate skills + job required skills into one corpus
# so the vectorizer learns a shared vocabulary

candidate_skills = candidates['Skills_Clean'].fillna('').tolist()
job_skills = jobs['Required_Skills_Clean'].fillna('').tolist()

all_skills_corpus = candidate_skills + job_skills

def split_skill_tokens(text):
    """Return comma-separated skills as normalized tokens."""
    return [s.strip() for s in str(text).split(',') if s.strip()]

# Fit TF-IDF on combined corpus
tfidf = TfidfVectorizer(
    tokenizer=split_skill_tokens,
    token_pattern=None,   # disable default pattern since we supply tokenizer
    lowercase=True
)
tfidf.fit(all_skills_corpus)

# Transform separately
candidate_tfidf = tfidf.transform(candidate_skills)
job_tfidf = tfidf.transform(job_skills)

print('Candidate TF-IDF matrix shape:', candidate_tfidf.shape)
print('Job TF-IDF matrix shape:', job_tfidf.shape)
print('Vocabulary size:', len(tfidf.vocabulary_))
print('Sample vocabulary:', list(tfidf.vocabulary_.keys())[:15])

# ══════════════════════════════════════════════════════════════════
# ENCODE EXPERIENCE LEVEL (numeric boost for matching)
# ══════════════════════════════════════════════════════════════════
level_map = {'Entry Level': 0, 'Mid Level': 1, 'Senior Level': 2, 'Unknown': 0}

candidates['Exp_Numeric'] = candidates['Experience_Level'].map(level_map).fillna(0)
jobs['Exp_Numeric'] = jobs['Experience Level'].map(level_map).fillna(0)



Candidate TF-IDF matrix shape: (815, 86)
Job TF-IDF matrix shape: (50000, 86)
Vocabulary size: 86
Sample vocabulary: ['python', 'machine learning', 'numpy', 'scikit-learn', 'sql', 'statistics', 'tensorflow', 'nlp', 'data visualization', 'pandas', 'pytho', 'nump', 'statistic', 'kubernetes', 'git']


In [5]:
# ══════════════════════════════════════════════════════════════════
# SAVE VECTORIZER + MATRICES
# ══════════════════════════════════════════════════════════════════
import scipy.sparse as sp

sp.save_npz(processed_dir / 'candidate_tfidf.npz', candidate_tfidf)
sp.save_npz(processed_dir / 'job_tfidf.npz', job_tfidf)

with open(processed_dir / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

candidates.to_csv(processed_dir / 'candidates_clean.csv', index=False)
jobs.to_csv(processed_dir / 'jobs_clean.csv', index=False)

print('\nFeature matrices and vectorizer saved.')


Feature matrices and vectorizer saved.
